# [Video RAG] 프레임 추출

In [ ]:
%pip install opencv-python pillow -Uq

In [ ]:
#구글드라이브 연동
from google.colab import drive
drive.mount('/content/drive') # 내 구글 드라이브를 /content/drive 경로로 마운트

BASE_PATH = '/content/drive/Mydrive/05_multimodal_rag/'  # 기본작업경로 변수

In [ ]:
import os  # os 모듈을 import한다.
from google.colab import userdata  # Colab의 userdata 모듈을 import한다.

os.environ['LANGSMITH_TRACING'] = 'true'                             # LANGSMITH_TRACING 환경 변수를 설정한다.
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com' # LANGSMITH의 엔드포인트를 설정한다.
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')  # LANGSMITH API 키를 환경 변수에 설정한다.
os.environ['LANGSMITH_PROJECT'] = 'skn23-multimodal-rag'             # LANGSMITH 프로젝트 이름을 설정한다.
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")        # OpenAI API 키를 환경 변수에 설정한다.

## opencv
**OpenCV**는 "Open Source Computer Vision Library"의 약자로, **실시간 이미지 및 영상 처리**를 위한 오픈소스 라이브러리이다. 인텔에서 개발을 시작했으며, 현재는 다양한 운영체제(Windows, Linux, macOS, Android 등)와 프로그래밍 언어(Python, C++, Java 등)를 지원한다.

**Python에서 OpenCV**는 주로 `cv2` 패키지를 통해 사용하며, 이미지와 영상의 읽기, 쓰기, 변환, 필터링, 객체 인식 등 다양한 컴퓨터 비전 기능을 쉽게 구현할 수 있다.

- **이미지/영상 처리**: 이미지 읽기, 쓰기, 변환, 필터 적용, 색상 변환 등 기본적인 영상 처리 기능을 제공한다.
- **컴퓨터 비전 알고리즘**: 얼굴 인식, 객체 추적, 패턴 분석 등 고급 컴퓨터 비전 기술을 구현할 수 있다.
- **머신러닝 지원**: OpenCV에는 머신러닝 관련 함수도 내장되어 있어, 이미지 분류, 객체 검출 등에 활용할 수 있다.
- **빠른 처리 속도**: 실시간 처리를 위해 최적화되어 있으며, C++ 기반으로 개발되어 빠른 연산이 가능하다.
- **활용 분야**:
  - 얼굴 인식, 객체 추적, 자율주행, 공장 자동화, 의료 영상 분석 등 **다양한 실생활 문제 해결**에 활용된다.

In [ ]:
!gdown 1zFdDejlTNGPSL5BVngJWTevOY2jipGrG

In [ ]:
import cv2
from google.colab.patches import cv2_imshow # Colab에서 OpenCV 이미지 출력용함수

# 로컬 실행
# img = cv2.imread('image.jpg')
# cv2.imshow('img' , 'img')
# cv2.waitkey(0)
# cv2.destroyALLWindows()

# colab 실행
img = cv2.imread('image.jpg')
cv2_imshow(img) # Colab 셀 출력 영역에 이미지 표시

In [ ]:
img2 = cv2.imread('image.jpg', cv2.IMREAD_GRAYSCALE)
cv2_imshow(img2) # Colab 셀 출력 영역에 이미지 표시

In [ ]:
# OpenCV로 단색 이미지 생성 및 저장  # 단색 이미지 생성 및 파일 저장 예제
import numpy as np  # 넘파이 라이브러리 불러오기
import cv2  # OpenCV 라이브러리 불러오기

# 검정색 이미지 (모든 채널값이 0)  # 검정색 이미지 생성
black = np.zeros((300, 300, 3), dtype = np.uint8)  # 300x300 크기의 검정 이미지 생성
cv2.imwrite('black.png' , black)                   # 'black.png' 파일로 이미지 저장

# 흰색 이미지 (모든 채널값이 255)  # 흰색 이미지 생성
white = np.ones((300, 300, 3), dtype = np.uint8) * 255  # 모든 값 255인 300x300 흰 이미지 생성
cv2.imwrite('white.png' , white)                        # 'white.png' 파일로 이미지 저장

# 빨간색 이미지 (B, G, R)에서 R만 255  # 빨간색 이미지 생성
red = np.zeros((300, 300, 3), dtype = np.uint8)   # 0으로 채워진 이미지 초기화
red[:, :, 2] = 255                                # 빨간색 채널만 255로 설정
cv2.imwrite('red.png' , red)                      # 'red.png' 파일로 이미지 저장

# 초록 이미지  # 초록색 이미지 생성
green = np.zeros((300, 300, 3), dtype = np.uint8)  # 0으로 채워진 이미지 초기화
green[:, :, 1] = 255                               # 초록색 채널만 255로 설정
cv2.imwrite('green.png' , green)                   # 'green.png' 파일로 이미지 저장

# 파랑 이미지  # 파란색 이미지 생성
blue = np.zeros((300, 300, 3), dtype = np.uint8)  # 0으로 채워진 이미지 초기화
blue[:, :, 0] = 255                               # 파란색 채널만 255로 설정
cv2.imwrite('blue.png' , blue)                    # 'blue.png' 파일로 이미지 저장

### 동영상 재성
- cap.read() : colab 재생시 콘솔에 프레임 연속출력
- HTML 태그 출력 : 동영상 html 태그를 통한 출력

In [ ]:
import cv2

video_path = os.path.join('/content/drive/MyDrive/05_multimodal_rag/videos/skiing.mp4.mp4')
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
  ret, frame = cap.read()

  if not ret:
    break
  cv2_imshow(frame)
cap.release()


### 썸네일 추출

In [ ]:
# HTML을 통한 비디오 출력
from IPython.display import HTML
from base64 import b64encode

video_path = os.path.join('/content/drive/MyDrive/05_multimodal_rag/videos/skiing2.mp4')
mp4 = open(video_path, 'rb').read()
b64_str = b64encode(mp4).decode()
data_url = f"data.video/mp4:base64, {b64_str}"

HTML(f"""
<video width='200' controls>
  <source src='{data_url}' type='video/mp4'>
<video>
""")

In [ ]:
from PIL import Image

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()
print(type(frame))

frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
img = Image.fromarray(frame)

print(f'{img.size = }')
print(f'{img.width = }')
print(f'{img.height = }')

img.thumbnail((480, 270))
img

In [ ]:
def show_video_summary(video_dir='videos', video_file = None, thumb_size = (240, 135)):
  assert video_file is not None, "video_file 인수는 반드시 지정해야 합니다."

  videopath = os.path.join(BASE_PATH, video_dir, video_file)
  cap = cv2.VideoCapture(video_path)
  if not cap.isOpened():
    print(f'{video_path} 파일이 없습니다.')
    return


  frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
  fps = cap.get(cv2.CAP_PROP_FPS)
  duration = frame_count / fps if fps > 0 else 0

  ret, frame = cap.read()  # 첫 프레임 1장 일기
  cap.release()   # 캡쳐 리소스 해제

  if not ret :
    print(f'썸네일 추출을 실패했습니다. {video_path}')
    thumbnail = None

  else :
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    thumbnail = Image.fromarray(frame_rgb)
    thumbnail.thumbnail(thumb_size)

  print(f'동영상 파일명 : {video_file}')
  print(f'총 프레임 수 : {frame_count}')
  print(f'프레임 레이트 : {fps:.2f}')
  print(f'동영상 길이 : {duration:.2f}')

  if thumbnail:
    display(thumbnail)

video_dir = '/content/drive/MyDrive/05_multimodal_rag/videos'
for video_file in os.listdir(os.path.join(BASE_PATH, video_dir)):
  if video_file.endswith('.mp4'):
    show_video_summary(video_dir, video_file, thumb_size = (480, 270))
    print(end = '\n\n')

- 프레임 추출

In [ ]:
import os
import cv2
from tqdm.notebook import tqdm

def extract_frame_from_videos(video_dir='videos', frame_dir='frames', frame_interval=1):
    video_dir_path = os.path.join('/content/drive/MyDrive/05_multimodal_rag/videos')
    frame_dir_path = os.path.join('/content/drive/MyDrive/05_multimodal_rag/frames')
    os.makedirs(frame_dir_path, exist_ok=True)

    video_files = [f for f in os.listdir(video_dir_path) if f.endswith('.mp4')]

    for idx_file, video_file in enumerate(video_files):
        video_path = os.path.join(video_dir_path, video_file)
        video_name = os.path.splitext(video_file)[0]
        save_subdir = os.path.join(frame_dir_path, video_name)
        os.makedirs(save_subdir, exist_ok=True)

        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        idx = 0
        saved_cnt = 0

        pbar = tqdm(total=frame_count, desc=video_name, leave=True, ncols=100)

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if idx % frame_interval == 0:
                frame_path = os.path.join(save_subdir, f'{video_name}_frame{idx:05d}.jpg')
                cv2.imwrite(frame_path, frame)
                saved_cnt += 1

            idx += 1
            pbar.update(1)

        cap.release()
        pbar.close()

        tqdm.write(f"{video_name}: 총 {frame_count} 프레임 중 {saved_cnt}개 저장")

    print('모든 동영상의 프레임 추출이 완료되었습니다.')

extract_frame_from_videos(frame_interval=25)